# ACTG 320 - ACTG 175

Here, we apply bridged comparison estimators for outcomes measured at a single time point. Data for this example comes from the AIDS Clinical Trial Group (ACTG). The first study is the ACTG-320 trial which compared triple-therapy to dual-therapy for prevention of disease progression among persons with HIV. The second study the ACTG-175 trial which compared dual-therapy to mono-therapy for prevention of disease progression among persons with HIV. We are interested in estimating the effect of triple-therapy relative to mono-therapy in the ACTG-320 source population. The outcome of interest is disease progression at 200 days post-randomization (note: the time-to-event aspect of the data is ignored, those right censored prior to 200 days have their outcome set as missing).

Here, we will illustrate the different estimators available.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cantilever.estimators.point import BridgeIPW, BridgeGComputation, BridgeAIPW

In [2]:
d = pd.read_csv("data/actg_discrete.csv")

## G-computation

The first estimator considered in the g-computation estimator. This estimator is based on modeling a single process, the outcome process, so only a single nuisance model is used. To start, we initialize the `BridgeGComputation` class with the previously harmonized data.

In [3]:
bgcomp = BridgeGComputation(data=d, outcome='y', action='art', sample='study', verbose=False)

The next step is to fit a model for the outcome. Since the outcome is binary, we will fit a logistic regression model.

In [4]:
bgcomp.outcome_model("C(art, contr.treatment(1)) + male + age + black + idu + C(karnof_cat)",
                     model_type='logistic')

After fitting this model, we can estimate the diagnostic based on the shared arms and the parameter of interest. The following code applies this estimation process for all these parameters.

In [5]:
bgcomp.estimate()

After estimation, we can examine the diagnostic based on the shared arm

In [6]:
bgcomp.diagnostics()

Outcome Regression Residuals
 * Skewed or large residual may indicate misspecification
--------------------------------------------------------------
      study=0  study=1
_                     
Mean    0.000    0.000
SD      0.145    0.299
Min    -0.996   -0.971
P25     0.006    0.035
P50     0.008    0.061
P75     0.030    0.131
Max     0.299    0.452

Shared Arm Diagnostic
No. Observations: 1969       | No. Input:        1969      
No. w/ Outcomes:  1734       | Outcome:          y         
Action:           art        | Sample:           study     
Outcome type:     Binary     | Model:            logistic  
Alpha:            0.05       | 
--------------------------------------------------------------
            Estimate    SE   LCL   UCL  P-value
                                               
Diagnostic      0.14  0.02  0.11  0.18      0.0
A=1,S=1         0.15  0.02  0.12  0.19      NaN
A=1,S=0         0.01  0.01 -0.00  0.02      NaN


As discussed further in Zivich et al. (2025), this is due to different eligibility criteria between the two trials. Specifically, the trials had different inclusion criteria by CD4, an important immunological indicator for disease progression in HIV. As ACTG-175 recruited individuals at a last advanced stage, dual-therapy looks like it is more protective.

We can mitigate some of this bias by restricting the data by baseline CD4 as also done in Zivich et al. (2025). Here, we apply that restriction and then re-estimate the diagnostic

In [7]:
ds = d.loc[(d['cd4'] >= 50) & (d['cd4'] <= 300)].copy()

In [8]:
bgcomp = BridgeGComputation(data=ds, outcome='y', action='art', sample='study', verbose=False)
bgcomp.outcome_model("C(art, contr.treatment(1)) + male + age + black + idu + C(karnof_cat)",
                     model_type='logistic')
bgcomp.estimate()
bgcomp.diagnostics()

Outcome Regression Residuals
 * Skewed or large residual may indicate misspecification
--------------------------------------------------------------
      study=0  study=1
_                     
Mean   -0.000    0.000
SD      0.188    0.225
Min    -0.995   -0.984
P25     0.007    0.026
P50     0.009    0.047
P75     0.050    0.070
Max     0.413    0.229

Shared Arm Diagnostic
No. Observations: 1034       | No. Input:        1034      
No. w/ Outcomes:  860        | Outcome:          y         
Action:           art        | Sample:           study     
Outcome type:     Binary     | Model:            logistic  
Alpha:            0.05       | 
--------------------------------------------------------------
            Estimate    SE   LCL   UCL  P-value
                                               
Diagnostic      0.07  0.02  0.03  0.10      0.0
A=1,S=1         0.08  0.02  0.05  0.11      NaN
A=1,S=0         0.01  0.01 -0.00  0.03      NaN


Here, we see the difference between the shared arms is somewhat mitigated, but it is still substantially different. This diagnostic results would suggest that this fusion of trials was not successful. Here, we would not provide the results for the comparison of interest as a result.

To showcase the functionality of the software, we set aside this implication and look at the differences comparing triple-therapy to mono-therapy.

In [9]:
bgcomp.summary()

Estimator:        Parametric G-computation
--------------------------------------------------------------
No. Observations: 1034       | No. Input:        1034      
No. w/ Outcomes:  860        | Outcome:          y         
Action:           art        | Sample:           study     
Outcome type:     Binary     | Model:            logistic  
Alpha:            0.05       | 
--------------------------------------------------------------
             Estimate    SE   LCL   UCL  P-value
                                                
Single-span     -0.08  0.04 -0.16 -0.01     0.03
Multi-span      -0.15  0.04 -0.23 -0.07     0.00
Diagnostic       0.07  0.02  0.03  0.10     0.00
A=2,S=1          0.03  0.01  0.01  0.05      NaN
A=1,S=1          0.08  0.02  0.05  0.11      NaN
A=1,S=0          0.01  0.01 -0.00  0.03      NaN
A=0,S=0          0.12  0.04  0.04  0.19      NaN


In [10]:
# Nuisance models: weights
action_model = "1"
sample_model = ""
censor_model = ""

# Nuisance models: outcome
outcome_model = "C(art, contr.treatment(1)) + male + age + black + idu + C(karnof_cat) + cd4"

## Inverse Probability Weighting

Next, we consider the inverse probability weighting estimator. To save time, we incorporate the previous restriction by baseline CD4 into this analysis. The bridge IPW estimator is instead based on three nuisance models for three different weights: inverse odds of sampling, inverse probability of treatment, and inverse probability of outcome missingness weights. These weights account for selection bias, confounding (in the case of observational data), and informative missing data, respectively. The estimator can be accessed via `BridgeIPW` and is initialized in the same way as the g-computation functionality

In [11]:
bipw = BridgeIPW(data=ds, outcome='y', action='art', sample='study', verbose=False)

Differences arise when specifying the nuisance models. Here, we need to specify three

In [12]:
# Sampling model for IOSW
bipw.sample_model("male + age + black + idu + C(karnof_cat)")
# Action model for IPTW
bipw.action_model("1")
# Missingness model for IPMW
bipw.missing_model("C(art) + male + age + black + idu + C(karnof_cat)")

In [13]:
bipw.estimate()

As before, we can examine the diagnostics

In [14]:
bipw.diagnostics()

Weight Diagnostics
Inverse Probability of Treatment Weights
 * The overall 'Mean' column should be near 2
 * Action-specific 'Sum' columns should be approximately equal
--------------------------------------------------------------
study = 0
--------------------------------------------------------------
         art   art=0   art=1
_                           
Mean    2.00    3.04    1.49
SD      0.73    0.00    0.00
Min     1.49    3.04    1.49
P25     1.49    3.04    1.49
P50     1.49    3.04    1.49
P75     3.04    3.04    1.49
Max     3.04    3.04    1.49
Sum   668.00  334.00  334.00
--------------------------------------------------------------
study = 1
--------------------------------------------------------------
          art   art=1   art=2
_                            
Mean     2.00    2.03    1.97
SD       0.03    0.00    0.00
Min      1.97    2.03    1.97
P25      1.97    2.03    1.97
P50      1.97    2.03    1.97
P75      2.03    2.03    1.97
Max      2.03    2.03    1.97

Again, we see a difference between the shared arms. Similarly, we examine the main results despite the diagnostic indicating this is invalid

In [15]:
bipw.summary()

Estimator:        Inverse Probability Weighting - Hajek
--------------------------------------------------------------
No. Observations: 1034       | No. Input:        1034      
No. w/ Outcomes:  860        | Outcome:          y         
Action:           art        | Sample:           study     
Outcome type:     Binary     | Model:            none      
Alpha:            0.05       | 
--------------------------------------------------------------
             Estimate    SE   LCL   UCL  P-value
                                                
Single-span     -0.08  0.04 -0.15 -0.01     0.02
Multi-span      -0.15  0.04 -0.23 -0.07     0.00
Diagnostic       0.07  0.02  0.03  0.10     0.00
A=2,S=1          0.03  0.01  0.01  0.05      NaN
A=1,S=1          0.08  0.02  0.05  0.11      NaN
A=1,S=0          0.01  0.01 -0.01  0.03      NaN
A=0,S=0          0.12  0.03  0.05  0.18      NaN


## Augmented Inverse Probability Weighting

The final estimator is the augmented inverse probability weighting (AIPW) estimator. This estimator can be viewed as a combination of the previous IPW and g-computation estimators. Here, the weighted-regression variation of AIPW is implemented due to its better finite-sample properties. To start, we initialize the `BridgeAIPW` class with the CD4-restricted data set.

In [16]:
baipw = BridgeAIPW(data=ds, outcome='y', action='art', sample='study', verbose=False)

For the AIPW estimator, all of the previous nuisance models must be fit.

In [17]:
# Sampling model for IOSW
baipw.sample_model("male + age + black + idu + C(karnof_cat)")
# Action model for IPTW
baipw.action_model("1")
# Missingness model for IPMW
baipw.missing_model("C(art) + male + age + black + idu + C(karnof_cat)")
# Outcome model
baipw.outcome_model("C(art, contr.treatment(1)) + male + age + black + idu + C(karnof_cat)",
                    model_type='logistic')

In [18]:
baipw.estimate()

Diagnostics for the AIPW estimator can be output via the `diagnostics` function. As these diagnostics are a combination of the previous IPW and g-computation diagnostics, we do not output them here.

The following are the summary results for the AIPW estimator

In [19]:
baipw.summary()

Estimator:        Augmented Inverse Probability Weighting
--------------------------------------------------------------
No. Observations: 1034       | No. Input:        1034      
No. w/ Outcomes:  860        | Outcome:          y         
Action:           art        | Sample:           study     
Outcome type:     Binary     | Model:            logistic  
Alpha:            0.05       | 
--------------------------------------------------------------
             Estimate    SE   LCL   UCL  P-value
                                                
Single-span     -0.07  0.04 -0.15 -0.00     0.04
Multi-span      -0.14  0.04 -0.23 -0.06     0.00
Diagnostic       0.07  0.02  0.03  0.11     0.00
A=2,S=1          0.03  0.01  0.01  0.05      NaN
A=1,S=1          0.08  0.02  0.05  0.11      NaN
A=1,S=0          0.01  0.01 -0.01  0.03      NaN
A=0,S=0          0.11  0.04  0.04  0.18      NaN


Again, a substantial difference for the shared arm diagnostic is observed.

## References

Hammer SM et al. (1996). A trial comparing nucleoside monotherapy with combination therapy in HIV-infected adults with CD4 cell counts from 200 to 500 per cubic millimeter. *New England Journal of Medicine*, 335(15):1081-1090.

Hammer SM, et al. (1997). A controlled trial of two nucleoside analogues plus indinavir in persons with human immunodeficiency virus infection and CD4 cell counts of 200 per cubic millimeter or less. *New England Journal of Medicine*, 337(11):725-733.

Shook-Sa BE, Zivich PN, Rosin SP, Edwards JK, Adimora AA, Hudgens MG, Cole SR. (2024). Fusing Trial Data for Treatment Comparisons: Single versus Multi-Span Bridging. *Statistics in Medicine*, 43(4):793-815.